In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

assert Path("/content/drive").is_dir(), "STOP: Google Drive mount is unavailable."
print("PASS: Google Drive mounted for this Colab runtime.")

## Stage 1 - Mount Google Drive and verify the known dataset locations

This stage connects the ephemeral Colab runtime to persistent Google Drive storage. Drive access and workbook-header compatibility have already been verified: Dataset A contains 12 readable XLSX workbooks with the `tweets` sheet and common documented 31-column header; Dataset B contains 70 readable XLSX workbooks with the `Sheet1` sheet and common documented 10-column header.

The next cell verifies only that the known directories remain visible and shows up to five directory entries from each. It does not open workbook contents.

**Storage:** persistent read-only dataset access under `/content/drive/MyDrive/...`.

**Stop conditions:** stop if Drive mounting fails, either dataset directory is absent, or either path is not a directory.

In [ ]:
from pathlib import Path

dataset_a = Path(
    "/content/drive/MyDrive/Thesis/Dataset A/core_army_pro_fans_tweets"
)

dataset_b = Path(
    "/content/drive/MyDrive/Thesis/Dataset B/statuses_data"
)

print("Dataset A exists:", dataset_a.exists())
print("Dataset B exists:", dataset_b.exists())

if dataset_a.exists():
    print("Dataset A sample:", list(dataset_a.iterdir())[:5])

if dataset_b.exists():
    print("Dataset B sample:", list(dataset_b.iterdir())[:5])

assert dataset_a.is_dir(), "STOP: Dataset A directory is not accessible."
assert dataset_b.is_dir(), "STOP: Dataset B directory is not accessible."
print("PASS: Verified Dataset A and Dataset B directory access.")

## Stage 2 - Define canonical ephemeral and persistent paths

`/content/community-evolution-modeling` is ephemeral Colab runtime storage. The repository source code must be cloned there and will disappear when the runtime is replaced.

`/content/drive/MyDrive/...` is persistent Google Drive storage. Datasets, manifests, reports, and checkpoints must remain in Google Drive.

Safety rules for this notebook:

- Never delete, overwrite, move, or modify the raw Dataset A or Dataset B directories.
- Never copy the complete raw datasets into the output directory.
- Removing an old repository clone under `/content` is allowed because it is ephemeral.
- Removing anything under `/content/drive` is forbidden.

**Storage:** path definition only; no files are changed.

**Stop conditions:** stop if either dataset path is not a directory or if the output root is not under `/content/drive/MyDrive`.

In [ ]:
from pathlib import Path

DATASET_A_ROOT = Path(
    "/content/drive/MyDrive/Thesis/Dataset A/core_army_pro_fans_tweets"
)

DATASET_B_ROOT = Path(
    "/content/drive/MyDrive/Thesis/Dataset B/statuses_data"
)

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Thesis/TDMEC_PROJECT_OUTPUTS"
)

REPO_ROOT = Path(
    "/content/community-evolution-modeling"
)

print("Dataset A:", DATASET_A_ROOT)
print("Dataset B:", DATASET_B_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Repository root:", REPO_ROOT)

assert DATASET_A_ROOT.is_dir(), "STOP: Dataset A root is not a directory."
assert DATASET_B_ROOT.is_dir(), "STOP: Dataset B root is not a directory."
assert str(OUTPUT_ROOT).startswith("/content/drive/MyDrive/"), (
    "STOP: Output root must remain in persistent Google Drive storage."
)
assert REPO_ROOT == Path("/content/community-evolution-modeling")
print("PASS: Canonical paths are configured safely.")

## Stage 3 - Clone and pin the audited repository commit

This stage removes only an existing ephemeral clone at the exact repository path, clones the public repository, checks out the audited commit in detached mode, and verifies both the commit and clean working tree.

**Storage:** ephemeral writes under `/content/community-evolution-modeling`; nothing under Google Drive is removed.

**Stop conditions:** stop if the removal target is not exactly the expected path under `/content`, if it is under `/content/drive`, if cloning or checkout fails, if the observed commit differs, or if the cloned working tree is not clean.

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/faezehmzf/community-evolution-modeling.git"
EXPECTED_SHA = "7e10748067e36190f025254fac42049b29d738f9"

ephemeral_root = Path("/content").resolve()
persistent_root = Path("/content/drive").resolve()
repository_target = REPO_ROOT.resolve(strict=False)

assert repository_target == Path("/content/community-evolution-modeling"), (
    "STOP: Refusing to remove an unexpected repository path."
)
assert repository_target.is_relative_to(ephemeral_root), (
    "STOP: Repository target must be under /content."
)
assert not repository_target.is_relative_to(persistent_root), (
    "STOP: Removing anything under /content/drive is forbidden."
)

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ["git", "clone", REPOSITORY_URL, str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", EXPECTED_SHA],
    cwd=REPO_ROOT,
    check=True,
)

observed_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()
working_tree_status = subprocess.check_output(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    text=True,
)

assert observed_sha == EXPECTED_SHA, (
    f"STOP: Expected commit {EXPECTED_SHA}, observed {observed_sha}."
)
assert working_tree_status == "", "STOP: Cloned working tree is not clean."

print("Observed commit:", observed_sha)
print("PASS: Repository cloned, pinned, and verified clean.")

## Stage 4 - Install the repository with the supported test extra

The audited `pyproject.toml` defines the `test` optional dependency and does not define a `dev` extra. The current notebook Python executable is used explicitly for both installation and package discovery.

**Storage:** editable package installation in the ephemeral Colab runtime.

**Stop conditions:** stop on any pip failure or if package discovery fails.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        f"{REPO_ROOT}[test]",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "show",
        "tdmec-discovery",
    ],
    check=True,
)

print("PASS: Installed the repository with the supported [test] extra.")

## Stage 5 - Verify imports, versions, module locations, and CLI help

The editable installation must resolve all four repository packages from the checked-out `src` directory. The Phase 1 and Phase 2 versions are fixed for the audited commit. CLI help is invoked with the current notebook Python executable.

**Storage:** read-only package and CLI validation in ephemeral runtime storage.

**Stop conditions:** stop on `ModuleNotFoundError`, `ImportError`, unexpected versions, modules resolving outside the checkout, or CLI failure.

In [ ]:
import subprocess
import sys
from pathlib import Path

import tdmec
import tdmec_diagnostics
import tdmec_discovery
import tdmec_pilot

packages = {
    "tdmec": tdmec,
    "tdmec_diagnostics": tdmec_diagnostics,
    "tdmec_discovery": tdmec_discovery,
    "tdmec_pilot": tdmec_pilot,
}
expected_source_root = (REPO_ROOT / "src").resolve()

for package_name, package in packages.items():
    package_file = Path(package.__file__).resolve()
    print(f"{package_name}.__file__:", package_file)
    assert package_file.is_relative_to(expected_source_root), (
        f"STOP: {package_name} resolved outside {expected_source_root}."
    )

assert tdmec.__version__ == "0.1.0-phase1", (
    f"STOP: Unexpected tdmec version: {tdmec.__version__}"
)
assert tdmec_diagnostics.__version__ == "0.2.0-phase2", (
    "STOP: Unexpected tdmec_diagnostics version: "
    f"{tdmec_diagnostics.__version__}"
)

subprocess.run(
    [sys.executable, "-m", "tdmec_diagnostics.cli", "--help"],
    cwd=REPO_ROOT,
    check=True,
)

print("tdmec version:", tdmec.__version__)
print("tdmec_diagnostics version:", tdmec_diagnostics.__version__)
print("PASS: Imports, versions, locations, and CLI help are verified.")

## Stage 6 - Run the complete repository test suite

The complete test suite is run from the pinned checkout with bytecode generation disabled and the pytest cache provider disabled. The audited local baseline was 123 passing tests, but this notebook does not fabricate or hard-code a Colab result: success is printed only after the actual pytest process exits with code zero.

**Storage:** tests use ephemeral runtime and temporary storage; no real datasets are accessed by the repository tests.

**Stop conditions:** stop immediately on collection failure, import failure, any failed test, or any nonzero pytest exit code.

In [ ]:
import os
import subprocess
import sys

test_environment = os.environ.copy()
test_environment["PYTHONDONTWRITEBYTECODE"] = "1"

test_process = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-p",
        "no:cacheprovider",
        "-v",
        "--tb=short",
    ],
    cwd=REPO_ROOT,
    env=test_environment,
    check=False,
)

assert test_process.returncode == 0, (
    f"STOP: Repository tests exited with code {test_process.returncode}."
)
print("PASS: Complete repository test suite exited successfully.")

## Stage 7 - Discover and select an existing node-map Parquet file

This read-only stage searches persistent Google Drive storage for likely node-map Parquet filenames. It never invokes `scripts/build_node_index_map.py`, never rebuilds the map from Dataset A, never reads Parquet rows, and never prints external account IDs.

Select the correct existing file by entering its displayed number.

**Storage:** persistent Google Drive read-only discovery.

**Stop conditions:** stop if no candidate exists, the selection is not a valid number, the number is outside the displayed range, or the selected path is not a file.

In [ ]:
from pathlib import Path

search_root = Path("/content/drive/MyDrive")
assert search_root.is_dir(), "STOP: /content/drive/MyDrive is not accessible."

candidate_paths = []
for path in search_root.rglob("*.parquet"):
    if not path.is_file():
        continue
    lower_name = path.name.lower()
    if (
        lower_name in {"node_map.parquet", "node_index_map.parquet"}
        or ("node" in lower_name and "map" in lower_name)
        or ("node" in lower_name and "index" in lower_name)
    ):
        candidate_paths.append(path)

NODE_MAP_CANDIDATES = sorted(set(candidate_paths), key=lambda path: str(path))
assert NODE_MAP_CANDIDATES, "STOP: No likely node-map Parquet file was found."

for number, candidate in enumerate(NODE_MAP_CANDIDATES, start=1):
    print(f"{number}. {candidate}")

selection_text = input("Enter the number of the validated node-map candidate: ").strip()
assert selection_text.isdigit(), "STOP: Selection must be a displayed integer."
selection_index = int(selection_text)
assert 1 <= selection_index <= len(NODE_MAP_CANDIDATES), (
    "STOP: Selected number is outside the displayed candidate range."
)

SELECTED_NODE_MAP = NODE_MAP_CANDIDATES[selection_index - 1]
assert SELECTED_NODE_MAP.is_file(), "STOP: Selected candidate is not a file."
print("PASS: Existing node-map candidate selected without modification.")

## Stage 8 - Validate the selected map with the repository loader

The audited repository interface is `load_node_map(...) -> NodeMap`. `NodeMap` is a dataclass with `mapping`, `min_index`, and `max_index`; it is not a pandas DataFrame.

**Storage:** selected persistent Parquet file is read but never modified.

**Stop conditions:** stop on any loader exception, a non-`NodeMap` return value, unexpected count, or unexpected index bounds.

In [ ]:
from tdmec_pilot.node_map import NodeMap, load_node_map

node_map = load_node_map(
    SELECTED_NODE_MAP,
    expected_count=16_736,
    index_min=0,
    index_max=16_735,
)

assert isinstance(node_map, NodeMap), "STOP: Loader did not return NodeMap."
assert len(node_map) == 16_736, "STOP: NodeMap entry count is not 16,736."
assert node_map.min_index == 0, "STOP: NodeMap minimum index is not 0."
assert node_map.max_index == 16_735, (
    "STOP: NodeMap maximum index is not 16,735."
)

print("PASS: Repository NodeMap validation succeeded.")

## Stage 9 - Run supplemental node-map invariants and record SHA-256

The current repository loader does not fully detect duplicate indices or missing interior indices. This stage reads only the two required columns with pandas, validates the complete frozen mapping, and calculates the selected file's SHA-256 with streaming reads.

No Parquet rows or external account IDs are printed.

**Storage:** persistent source file is read-only; validation data exists only in runtime memory.

**Stop conditions:** stop on missing columns, incorrect row count, nulls, unsafe author-ID representation, noninteger index dtype, malformed IDs, duplicates, incomplete index set, noncanonical index order, noncanonical numeric author-ID order, or hash-read failure. Do not rebuild automatically.

In [ ]:
import hashlib

import pandas as pd
from pandas.api.types import is_bool_dtype, is_integer_dtype, is_numeric_dtype

required_node_map_columns = ["author_account_id", "node_index"]
node_map_frame = pd.read_parquet(SELECTED_NODE_MAP)
assert set(required_node_map_columns).issubset(node_map_frame.columns), (
    "STOP: Node-map Parquet lacks required columns."
)

validation_frame = node_map_frame[required_node_map_columns].copy()
author_values = validation_frame["author_account_id"]
index_values = validation_frame["node_index"]

assert len(validation_frame) == 16_736, "STOP: Expected exactly 16,736 rows."
assert not author_values.isna().any(), "STOP: Null author account ID found."
assert not index_values.isna().any(), "STOP: Null node index found."
assert not is_numeric_dtype(author_values.dtype), (
    "STOP: author_account_id has a numeric dtype and may be precision-lossy."
)
assert author_values.map(lambda value: isinstance(value, str)).all(), (
    "STOP: Every author_account_id must remain string data."
)
assert is_integer_dtype(index_values.dtype) and not is_bool_dtype(index_values.dtype), (
    "STOP: node_index must have an integer dtype."
)

author_id_strings = author_values.astype(str)
node_indices = index_values.astype(int)

assert author_id_strings.str.fullmatch(r"\d+").all(), (
    "STOP: Every author account ID must be an integer-like digit string."
)
assert author_id_strings.nunique() == 16_736, (
    "STOP: Author account IDs are not unique."
)
assert node_indices.nunique() == 16_736, "STOP: Node indices are not unique."
assert set(node_indices) == set(range(16_736)), (
    "STOP: Node-index set is not exactly 0 through 16,735."
)

ordered_by_index = validation_frame.assign(
    author_account_id=author_id_strings,
    node_index=node_indices,
).sort_values("node_index")
assert ordered_by_index["node_index"].tolist() == list(range(16_736)), (
    "STOP: Sorting by node_index did not produce canonical index order."
)
assert ordered_by_index["author_account_id"].tolist() == sorted(
    author_id_strings.tolist(),
    key=int,
), "STOP: Numeric author-ID order does not match node-index order."

def streaming_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

NODE_MAP_SHA256 = streaming_sha256(SELECTED_NODE_MAP)
NODE_MAP_ROW_COUNT = len(validation_frame)
NODE_MAP_UNIQUE_AUTHORS = author_id_strings.nunique()
NODE_MAP_UNIQUE_INDICES = node_indices.nunique()

print("Row count:", NODE_MAP_ROW_COUNT)
print("Unique author-ID count:", NODE_MAP_UNIQUE_AUTHORS)
print("Unique node-index count:", NODE_MAP_UNIQUE_INDICES)
print("Minimum index:", int(node_indices.min()))
print("Maximum index:", int(node_indices.max()))
print("SHA-256:", NODE_MAP_SHA256)
print("PASS: Supplemental node-map validation succeeded.")

## Stage 10 - Publish the validated node map and privacy-safe validation manifest

Only after both validation stages pass, the selected map is copied to the canonical persistent path. The source file is never modified. Existing canonical artifacts are accepted only when their content exactly matches; a conflicting existing file causes a hard stop rather than overwrite.

The JSON manifest contains only approved aggregate validation facts and filenames. It contains no external account IDs, usernames, raw text, email addresses, credentials, or private absolute Drive paths.

**Storage:** persistent writes under `TDMEC_PROJECT_OUTPUTS/manifests`; raw dataset directories remain untouched.

**Stop conditions:** stop if an existing canonical map has a different hash, copying fails, the copied hash differs, manifest privacy validation fails, or an existing validation manifest differs from the expected content.

In [ ]:
import json
import shutil

from tdmec_diagnostics.privacy import assert_privacy_safe_mapping

MANIFESTS_ROOT = OUTPUT_ROOT / "manifests"
CANONICAL_NODE_MAP = MANIFESTS_ROOT / "node_index_map.parquet"
NODE_MAP_VALIDATION_MANIFEST = (
    MANIFESTS_ROOT / "node_index_map_validation_manifest.json"
)

MANIFESTS_ROOT.mkdir(parents=True, exist_ok=True)

same_source_and_target = (
    SELECTED_NODE_MAP.resolve() == CANONICAL_NODE_MAP.resolve(strict=False)
)
if CANONICAL_NODE_MAP.exists():
    existing_hash = streaming_sha256(CANONICAL_NODE_MAP)
    assert existing_hash == NODE_MAP_SHA256, (
        "STOP: Existing canonical node map has a different SHA-256; "
        "the notebook will not overwrite it."
    )
elif not same_source_and_target:
    shutil.copy2(SELECTED_NODE_MAP, CANONICAL_NODE_MAP)

assert CANONICAL_NODE_MAP.is_file(), "STOP: Canonical node map was not published."
canonical_hash = streaming_sha256(CANONICAL_NODE_MAP)
assert canonical_hash == NODE_MAP_SHA256, (
    "STOP: Canonical node-map hash differs from the validated source hash."
)

node_map_validation_record = {
    "artifact_type": "frozen_node_index_map_validation",
    "sha256": NODE_MAP_SHA256,
    "row_count": NODE_MAP_ROW_COUNT,
    "columns": required_node_map_columns,
    "index_min": int(node_indices.min()),
    "index_max": int(node_indices.max()),
    "unique_author_ids": NODE_MAP_UNIQUE_AUTHORS,
    "unique_node_indices": NODE_MAP_UNIQUE_INDICES,
    "exact_index_set_0_to_16735": True,
    "canonical_numeric_author_id_order": True,
    "source_filename": SELECTED_NODE_MAP.name,
    "canonical_filename": CANONICAL_NODE_MAP.name,
    "audited_repository_commit": EXPECTED_SHA,
}
approved_manifest_fields = {
    "artifact_type",
    "sha256",
    "row_count",
    "columns",
    "index_min",
    "index_max",
    "unique_author_ids",
    "unique_node_indices",
    "exact_index_set_0_to_16735",
    "canonical_numeric_author_id_order",
    "source_filename",
    "canonical_filename",
    "audited_repository_commit",
}
assert set(node_map_validation_record) == approved_manifest_fields
assert_privacy_safe_mapping(node_map_validation_record)

if NODE_MAP_VALIDATION_MANIFEST.exists():
    existing_manifest = json.loads(
        NODE_MAP_VALIDATION_MANIFEST.read_text(encoding="utf-8")
    )
    assert existing_manifest == node_map_validation_record, (
        "STOP: Existing validation manifest differs; it will not be overwritten."
    )
else:
    with NODE_MAP_VALIDATION_MANIFEST.open("x", encoding="utf-8") as stream:
        json.dump(node_map_validation_record, stream, indent=2, sort_keys=True)
        stream.write("\n")

print("Canonical node-map filename:", CANONICAL_NODE_MAP.name)
print("SHA-256:", canonical_hash)
print("PASS: Canonical node map and privacy-safe validation manifest published.")

## Stage 11 - Discover the exact workbook set and select fixed smoke inputs

This stage deterministically discovers XLSX files directly from the two known dataset roots. It does not reopen every workbook because the workbook and header preflight has already completed successfully.

All 82 basenames must be unique because the audited checkpoint implementation uses basename file references. The smoke inputs are fixed to `core_army_pro_fans_tweets_part_001.xlsx` and `statuses-0.xlsx`.

**Storage:** persistent raw directories are listed read-only.

**Stop conditions:** stop unless counts are exactly 12 and 70, if any basename collision exists, or if either exact smoke workbook is absent.

In [ ]:
from pathlib import Path

def discover_xlsx_files(root: Path):
    assert root.is_dir(), f"STOP: Dataset root is not a directory: {root}"
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() == ".xlsx"
    )

DATASET_A_FILES = discover_xlsx_files(DATASET_A_ROOT)
DATASET_B_FILES = discover_xlsx_files(DATASET_B_ROOT)

assert len(DATASET_A_FILES) == 12, (
    f"STOP: Expected 12 Dataset A workbooks, found {len(DATASET_A_FILES)}."
)
assert len(DATASET_B_FILES) == 70, (
    f"STOP: Expected 70 Dataset B workbooks, found {len(DATASET_B_FILES)}."
)

all_workbook_basenames = [
    path.name for path in DATASET_A_FILES + DATASET_B_FILES
]
assert len(all_workbook_basenames) == len(set(all_workbook_basenames)), (
    "STOP: Workbook basename collision detected across Dataset A and B."
)

SMOKE_DATASET_A_FILE = DATASET_A_ROOT / (
    "core_army_pro_fans_tweets_part_001.xlsx"
)
SMOKE_DATASET_B_FILE = DATASET_B_ROOT / "statuses-0.xlsx"

assert SMOKE_DATASET_A_FILE in DATASET_A_FILES, (
    "STOP: Exact Dataset A smoke workbook is absent."
)
assert SMOKE_DATASET_B_FILE in DATASET_B_FILES, (
    "STOP: Exact Dataset B smoke workbook is absent."
)

print("Dataset A file count:", len(DATASET_A_FILES))
print("Dataset B file count:", len(DATASET_B_FILES))
print("Smoke Dataset A basename:", SMOKE_DATASET_A_FILE.name)
print("Smoke Dataset B basename:", SMOKE_DATASET_B_FILE.name)
print("PASS: Exact workbook counts, basename uniqueness, and smoke inputs verified.")

## Stage 12 - Execute the one-file-per-dataset real-data smoke run

This stage uses only the audited public interfaces `load_diagnostics_config(path=None)` and `run_diagnostics(...) -> Dict[str, Any]`. It passes explicit file lists, so no dataset directory is copied into a temporary source cache. It also omits `checkpoint_root` because explicit checkpoint-root handling is defective in real mode at the audited commit.

A unique UTC timestamp creates a new persistent smoke-run directory. The configuration uses `resume_mode="restart"` only for this new, asserted-absent run ID.

**Storage:** raw workbooks are read-only; reports and checkpoints are written persistently under `OUTPUT_ROOT/diagnostics/<run_id>`.

**Stop conditions:** stop on any exception, OOM, disconnect, adapter or schema failure, incomplete result, false real-data flag, or status other than `REVIEW_REQUIRED`.

In [ ]:
from dataclasses import replace
from datetime import datetime, timezone

from tdmec_diagnostics.config import load_diagnostics_config
from tdmec_diagnostics.pipeline import run_diagnostics

smoke_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SMOKE_RUN_ID = f"phase2-real-smoke-a1-b1-{smoke_timestamp}"
SMOKE_RUN_DIR = OUTPUT_ROOT / "diagnostics" / SMOKE_RUN_ID

assert not SMOKE_RUN_DIR.exists(), (
    "STOP: Generated smoke-run directory already exists; no existing run "
    "will be overwritten."
)

smoke_config = replace(
    load_diagnostics_config(REPO_ROOT / "configs" / "phase2_diagnostics.yaml"),
    resume_mode="restart",
)

smoke_result = run_diagnostics(
    output_root=OUTPUT_ROOT,
    config=smoke_config,
    mode="real",
    run_id=SMOKE_RUN_ID,
    dataset_a_files=[SMOKE_DATASET_A_FILE],
    dataset_b_files=[SMOKE_DATASET_B_FILE],
    node_index_map=CANONICAL_NODE_MAP,
)

assert isinstance(smoke_result, dict), "STOP: run_diagnostics did not return dict."
assert smoke_result["complete"] is True, "STOP: Smoke run is incomplete."
assert smoke_result["manifest"]["real_data_executed"] is True, (
    "STOP: Smoke result does not confirm real-data adapter execution."
)
assert smoke_result["status"] == "REVIEW_REQUIRED", (
    f"STOP: Unexpected smoke status: {smoke_result['status']}"
)

print("Run ID:", smoke_result["run_id"])
print("Status:", smoke_result["status"])
print("Complete:", smoke_result["complete"])
print("Real data executed:", smoke_result["manifest"]["real_data_executed"])
print("Layout:", smoke_result["layout"])
print("PASS: One-file Dataset A plus one-file Dataset B smoke run completed.")

## Stage 13 - Audit the exact smoke artifact set and safety invariants

A successful checkpoint-enabled smoke run must contain exactly 15 files. This stage validates the file set, report statuses, scientific hashes, manifest hashes, privacy-safe mappings, absence of known frozen external account IDs, checkpoint consistency, and post-run source checksums.

Only aggregate, privacy-safe report sections are printed. Candidate warnings are displayed for human review and are not automatic PASS conditions.

**Storage:** persistent reports and checkpoints are read; raw workbooks are rehashed read-only to verify immutability.

**Stop conditions:** stop on a missing or unexpected artifact, scientific hash mismatch, source checksum mismatch, privacy failure, raw external-ID leak, hard failure, checkpoint inconsistency, unexpected certification claim, or any unexplained invariant failure.

In [ ]:
import json
import re
from collections.abc import Mapping

from tdmec.hashing import sha256_file
from tdmec_diagnostics.privacy import assert_privacy_safe_mapping
from tdmec_diagnostics.reports import scientific_content_hash

REPORT_NAMES = [
    "calendar_report",
    "dedup_report",
    "text_length_report",
    "coverage_report",
    "warnings_and_failures",
    "unresolved_decision_evidence",
    "run_summary",
]
HUMAN_REPORT_NAMES = [
    "calendar_report",
    "dedup_report",
    "text_length_report",
    "coverage_report",
    "run_summary",
]
EXPECTED_SMOKE_ARTIFACTS = {
    "execution_manifest.json",
    "checkpoints/diagnostics_checkpoint.json",
    "checkpoints/accumulator_state.json",
    *{f"reports/{name}.json" for name in REPORT_NAMES},
    *{f"human/{name}.md" for name in HUMAN_REPORT_NAMES},
}

actual_smoke_artifacts = {
    path.relative_to(SMOKE_RUN_DIR).as_posix()
    for path in SMOKE_RUN_DIR.rglob("*")
    if path.is_file()
}
assert actual_smoke_artifacts == EXPECTED_SMOKE_ARTIFACTS, {
    "missing": sorted(EXPECTED_SMOKE_ARTIFACTS - actual_smoke_artifacts),
    "unexpected": sorted(actual_smoke_artifacts - EXPECTED_SMOKE_ARTIFACTS),
}

execution_manifest = json.loads(
    (SMOKE_RUN_DIR / "execution_manifest.json").read_text(encoding="utf-8")
)
reports = {
    name: json.loads(
        (SMOKE_RUN_DIR / "reports" / f"{name}.json").read_text(
            encoding="utf-8"
        )
    )
    for name in REPORT_NAMES
}
checkpoint = json.loads(
    (
        SMOKE_RUN_DIR / "checkpoints" / "diagnostics_checkpoint.json"
    ).read_text(encoding="utf-8")
)
accumulator_state = json.loads(
    (SMOKE_RUN_DIR / "checkpoints" / "accumulator_state.json").read_text(
        encoding="utf-8"
    )
)

expected_smoke_basenames = {
    SMOKE_DATASET_A_FILE.name,
    SMOKE_DATASET_B_FILE.name,
}
assert execution_manifest["processing_status"] == "REVIEW_REQUIRED"
assert execution_manifest["real_data_executed"] is True
assert execution_manifest["certification_claim"] is None
assert execution_manifest["hard_failure_counts"] == 0
assert execution_manifest["resume_state"]["n_files_tracked"] == 2
assert set(execution_manifest["resume_state"]["files_complete"]) == (
    expected_smoke_basenames
)

run_summary = reports["run_summary"]
assert run_summary["phase3_plus_implemented"] is False
assert run_summary["real_data_executed"] is True
assert run_summary["certification_claim"] is None
assert reports["coverage_report"]["node_universe_size"] == 16_736

allowed_statuses = {
    "UNVALIDATED",
    "DIAGNOSTIC_COMPLETE",
    "REVIEW_REQUIRED",
}
known_external_account_ids = set(node_map.mapping)

def contains_known_external_id(value) -> bool:
    if isinstance(value, Mapping):
        return any(contains_known_external_id(item) for item in value.values())
    if isinstance(value, (list, tuple)):
        return any(contains_known_external_id(item) for item in value)
    if isinstance(value, str):
        digit_tokens = re.findall(r"\d{6,}", value)
        return any(token in known_external_account_ids for token in digit_tokens)
    return False

for report_name, report in reports.items():
    assert report["status"] in allowed_statuses, (
        f"STOP: Unexpected status in {report_name}."
    )
    assert "CERTIFIED" not in report["status"].upper(), (
        f"STOP: Certification status found in {report_name}."
    )
    assert report.get("certification_claim") in (None, ""), (
        f"STOP: Unexpected certification claim in {report_name}."
    )

    report_without_self_hash = {
        key: value
        for key, value in report.items()
        if key != "scientific_content_hash"
    }
    assert report["scientific_content_hash"] == scientific_content_hash(
        report_without_self_hash
    ), f"STOP: Scientific self-hash mismatch in {report_name}."
    assert execution_manifest["report_content_hashes"][report_name] == (
        scientific_content_hash(report)
    ), f"STOP: Manifest report hash mismatch in {report_name}."

    assert_privacy_safe_mapping(report)
    assert not contains_known_external_id(report), (
        f"STOP: Raw external account identifier detected in {report_name}."
    )

shareable_manifest_fields = {
    key: value
    for key, value in execution_manifest.items()
    if key != "runtime_environment"
}
assert_privacy_safe_mapping(shareable_manifest_fields)
assert not contains_known_external_id(shareable_manifest_fields), (
    "STOP: Raw external account identifier detected in shareable manifest fields."
)

assert checkpoint["config_hash"] == accumulator_state["config_hash"], (
    "STOP: Checkpoint and accumulator config hashes differ."
)
assert set(checkpoint["files"]) == expected_smoke_basenames
assert all(item["complete"] for item in checkpoint["files"].values()), (
    "STOP: One or more checkpoint files are incomplete."
)
for component_name in ("calendar", "dedup", "text_length", "coverage"):
    assert set(accumulator_state[component_name]["files_seen"]) == (
        expected_smoke_basenames
    ), f"STOP: Accumulator file set differs for {component_name}."

manifest_sources = {
    item["file_ref"]: item
    for item in execution_manifest["source_file_identifiers"]
}
assert set(manifest_sources) == expected_smoke_basenames
for source_workbook in (SMOKE_DATASET_A_FILE, SMOKE_DATASET_B_FILE):
    assert sha256_file(source_workbook) == manifest_sources[
        source_workbook.name
    ]["checksum"], f"STOP: Source checksum changed for {source_workbook.name}."

warnings_report = reports["warnings_and_failures"]
assert warnings_report["hard_failure_counts"] == 0, (
    "STOP: Smoke reports contain a hard failure."
)

privacy_safe_review_summary = {
    "calendar_recommendation": reports["calendar_report"]["recommendation"],
    "calendar_reason_counts": reports["calendar_report"]["reason_counts"],
    "dedup_candidate_signatures": reports["dedup_report"][
        "candidate_signatures"
    ],
    "coverage_candidate_warnings": reports["coverage_report"][
        "candidate_warnings"
    ],
    "warning_counts": warnings_report["warning_counts"],
    "hard_failure_counts": warnings_report["hard_failure_counts"],
    "unresolved_decision_items": reports[
        "unresolved_decision_evidence"
    ]["items"],
}
assert_privacy_safe_mapping(privacy_safe_review_summary)
print(json.dumps(privacy_safe_review_summary, indent=2, sort_keys=True))
print("PASS: Exact artifacts, privacy, hashes, checksums, and checkpoints verified.")
print("REVIEW REQUIRED: Inspect every displayed candidate warning before proceeding.")

## Stage 14 - Validate sealed-run idempotent resume

This test validates only idempotent resume of an already completed and sealed run. It hashes all 15 artifacts, reloads the same scientific configuration with `resume_mode="resume"`, reruns the exact same inputs and run ID, and requires the pipeline's `resumed_from_sealed` marker plus unchanged artifact hashes.

It does **not** prove safe recovery from a disconnect inside a workbook, a crash between checkpoint and accumulator writes, an inconsistent partial transaction, or corrupted accumulator state.

**Storage:** persistent smoke artifacts are read; a correct sealed resume performs no artifact mutation.

**Stop conditions:** stop if the run is incomplete, `resumed_from_sealed` is not true, the artifact set changes, any artifact hash changes, or any count-bearing artifact drifts.

In [ ]:
from dataclasses import replace

artifact_hashes_before_resume = {
    path.relative_to(SMOKE_RUN_DIR).as_posix(): sha256_file(path)
    for path in SMOKE_RUN_DIR.rglob("*")
    if path.is_file()
}
assert set(artifact_hashes_before_resume) == EXPECTED_SMOKE_ARTIFACTS

resume_config = replace(
    load_diagnostics_config(REPO_ROOT / "configs" / "phase2_diagnostics.yaml"),
    resume_mode="resume",
)
resumed_result = run_diagnostics(
    output_root=OUTPUT_ROOT,
    config=resume_config,
    mode="real",
    run_id=SMOKE_RUN_ID,
    dataset_a_files=[SMOKE_DATASET_A_FILE],
    dataset_b_files=[SMOKE_DATASET_B_FILE],
    node_index_map=CANONICAL_NODE_MAP,
)

assert resumed_result["complete"] is True, "STOP: Sealed resume is incomplete."
assert resumed_result.get("resumed_from_sealed") is True, (
    "STOP: Pipeline did not report resumed_from_sealed=True."
)

artifact_hashes_after_resume = {
    path.relative_to(SMOKE_RUN_DIR).as_posix(): sha256_file(path)
    for path in SMOKE_RUN_DIR.rglob("*")
    if path.is_file()
}
assert artifact_hashes_after_resume == artifact_hashes_before_resume, (
    "STOP: Artifact drift or double counting detected after sealed resume."
)

print("SEALED RESUME: PASS")
print("No artifact drift.")
print("No double counting detected.")

## Stage 15 - Hard stop before full execution

This notebook intentionally contains no executable, disabled, or hidden full 12-plus-70 run call. Full execution remains blocked at audited commit `7e10748067e36190f025254fac42049b29d738f9` because of:

1. Unbounded Python memory growth in `DedupAccumulator`.
2. Very large serialized accumulator state.
3. File-level rather than chunk-level checkpointing.
4. A workbook interrupted midway restarts from row 1.
5. Nontransactional updates between `diagnostics_checkpoint.json` and `accumulator_state.json`.
6. A crash window that can cause silent undercounting.
7. Defective explicit checkpoint-root handling in real mode.
8. Incomplete evidence for QCAL-B01.
9. Incomplete evidence for QDEDUP-B01.
10. Incomplete Layer-2 threshold-review evidence.

Successful installation, tests, node-map validation, smoke execution, artifact audit, and sealed resume do not make the full run safe. They also do not authorize or implement Phase 3.

**Storage:** status reporting only; no files are changed.

**Stop condition:** always stop after the next cell. Do not proceed to a full Dataset A plus Dataset B execution.

In [ ]:
FULL_RUN_ALLOWED = False

assert FULL_RUN_ALLOWED is False, "STOP: Full-run safety gate must remain false."
print("Audited repository commit:", EXPECTED_SHA)
print("Smoke run ID:", SMOKE_RUN_ID)
print("Smoke run directory:", SMOKE_RUN_DIR)
print("Node-map SHA-256:", NODE_MAP_SHA256)
print("Full 12+70 run allowed:", FULL_RUN_ALLOWED)
print("PASS: Hard stop is active.")
print("FINAL STATUS: STOP BEFORE FULL RUN")